<a href="https://colab.research.google.com/github/patrickkirbyjr/DataPat/blob/main/wf_lead.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Compare Each Row to the Next Row using SQL (Python + DuckDB)

### Scenario
Oh boy, a new email! Looks like your boss just sent over a massive table of NYC taxi trips with this message:

> "I need to see how long each cab sits idle waiting for its next passenger. Can you pull the next trip's pickup time onto each row and flag whether the turnaround was Quick (< 1 min), Moderate (1-5 min), or Long (> 5 min) using SQL?"

In [1]:
#libraries, data, preview
import duckdb
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet"
df_preview = duckdb.sql(f"SELECT * FROM '{url}' LIMIT 5").df()
df_preview

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,2,2023-01-01 00:32:10,2023-01-01 00:40:36,1.0,0.97,1.0,N,161,141,2,9.3,1.00,0.5,0.00,0.0,1.0,14.30,2.5,0.00
1,2,2023-01-01 00:55:08,2023-01-01 01:01:27,1.0,1.10,1.0,N,43,237,1,7.9,1.00,0.5,4.00,0.0,1.0,16.90,2.5,0.00
2,2,2023-01-01 00:25:04,2023-01-01 00:37:49,1.0,2.51,1.0,N,48,238,1,14.9,1.00,0.5,15.00,0.0,1.0,34.90,2.5,0.00
3,1,2023-01-01 00:03:48,2023-01-01 00:13:25,0.0,1.90,1.0,N,138,7,1,12.1,7.25,0.5,0.00,0.0,1.0,20.85,0.0,1.25
4,2,2023-01-01 00:10:29,2023-01-01 00:21:19,1.0,1.43,1.0,N,107,79,1,11.4,1.00,0.5,3.28,0.0,1.0,19.68,2.5,0.00


In [7]:
# write query
query = f"""
WITH next_trips AS (
  SELECT
    VendorID,
    tpep_pickup_datetime,
    LEAD(tpep_pickup_datetime, 1) OVER (
      PARTITION BY VendorID
      ORDER BY tpep_pickup_datetime, tpep_dropoff_datetime
    ) AS next_pickup_datetime
  FROM '{url}'
  WHERE tpep_pickup_datetime >= '2023-01-01'
    AND tpep_pickup_datetime < '2023-01-02'
    AND trip_distance > 0
    AND PULocationID = 132
)
SELECT
  *,
  DATE_DIFF('minute', tpep_pickup_datetime, next_pickup_datetime) AS minutes_until_next_pickup,
  CASE
    WHEN next_pickup_datetime IS NULL THEN 'Last Trip'
    WHEN DATE_DIFF('minute', tpep_pickup_datetime, next_pickup_datetime) < 1 THEN 'Quick (<1m)'
    WHEN DATE_DIFF('minute', tpep_pickup_datetime, next_pickup_datetime) <= 5 THEN 'Moderate (1-5m)'
    ELSE 'Long (>5m)'
  END AS turnaround_flag
FROM next_trips
ORDER BY VendorID, tpep_pickup_datetime
LIMIT 15
"""
df = duckdb.sql(query).df()
df

,VendorID,tpep_pickup_datetime,next_pickup_datetime,minutes_until_next_pickup,turnaround_flag
0,1,2023-01-01 00:02:29,2023-01-01 00:07:31,5,Moderate (1-5m)
1,1,2023-01-01 00:07:31,2023-01-01 00:09:02,2,Moderate (1-5m)
2,1,2023-01-01 00:09:02,2023-01-01 00:11:44,2,Moderate (1-5m)
3,1,2023-01-01 00:11:44,2023-01-01 00:11:49,0,Quick (<1m)
4,1,2023-01-01 00:11:49,2023-01-01 00:13:30,2,Moderate (1-5m)
5,1,2023-01-01 00:13:30,2023-01-01 00:15:05,2,Moderate (1-5m)
6,1,2023-01-01 00:15:05,2023-01-01 00:16:10,1,Moderate (1-5m)
7,1,2023-01-01 00:16:10,2023-01-01 00:18:16,2,Moderate (1-5m)
8,1,2023-01-01 00:18:16,2023-01-01 00:18:20,0,Quick (<1m)
9,1,2023-01-01 00:18:20,2023-01-01 00:18:33,0,Quick (<1m)
